In [ ]:
import gradio as gr
import cv2
import mediapipe as mp
import numpy as np
import time
import threading
import math
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import LaserScan
from geometry_msgs.msg import Twist
from robot_wrapper import Robot 

# ==========================================
# 1. GLOBAL STATE & MEDIAPIPE SETUP
# ==========================================
global_robot_command = "NO HAND DETECTED"
thread_active = True

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ==========================================
# 2. GESTURE EVALUATION
# ==========================================
def evaluate_gesture(landmarks):
    wrist = landmarks[0]
    
    def is_extended(tip_idx, knuckle_idx):
        dist_tip = math.hypot(landmarks[tip_idx].x - wrist.x, landmarks[tip_idx].y - wrist.y)
        dist_knuckle = math.hypot(landmarks[knuckle_idx].x - wrist.x, landmarks[knuckle_idx].y - wrist.y)
        return dist_tip > dist_knuckle * 1.15
        
    index_ext = is_extended(8, 5)
    middle_ext = is_extended(12, 9)
    ring_ext = is_extended(16, 13)
    pinky_ext = is_extended(20, 17)
    
    open_fingers = sum([index_ext, middle_ext, ring_ext, pinky_ext])
    
    thumb_ext = math.hypot(landmarks[4].x - wrist.x, landmarks[4].y - wrist.y) > \
                math.hypot(landmarks[2].x - wrist.x, landmarks[2].y - wrist.y) * 1.1

    if open_fingers <= 1 and not thumb_ext: 
        return "STOP"
    if open_fingers <= 1 and thumb_ext: 
        return "BACKWARD"
        
    if open_fingers >= 3:
        fingertips_x = (landmarks[8].x + landmarks[12].x + landmarks[16].x + landmarks[20].x) / 4.0
        diff_x = fingertips_x - wrist.x
        
        if diff_x < -0.05: return "RIGHT"
        if diff_x > 0.05: return "LEFT"
        return "GO"
        
    return "HAND DETECTED"

def process_frame(image):
    global global_robot_command
    
    if image is None:
        return None
        
    # Gradio passes RGB images, which MediaPipe expects
    results = hands.process(image)
    
    current_gesture = "NO HAND DETECTED"
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Draw the skeleton on the frame
            mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            # Evaluate the gesture based on landmark coordinates
            current_gesture = evaluate_gesture(hand_landmarks.landmark)
            
    # Update the global command for the ROS2 thread
    global_robot_command = current_gesture
    
    # Overlay the text on the webcam feed
    cv2.putText(image, f"Command: {current_gesture}", (10, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
    
    return image

# ==========================================
# 3. ROS2 ROBOT CONTROL THREAD
# ==========================================
class LidarObserver(Node):
    def __init__(self, robot_instance):
        super().__init__('lidar_observer')
        self.robot = robot_instance 
        self.subscription = self.create_subscription(LaserScan, '/scan', self.avoiding_obstacles, 10)
        self.safety_latch_engaged = False

    def avoiding_obstacles(self, msg):
        ranges = msg.ranges
        if not ranges: return

        total_points = len(ranges)
        cone_size = max(1, int(total_points / 12)) 
        front_cone = ranges[:cone_size] + ranges[-cone_size:]
        
        valid_distances = [r for r in front_cone if 0.15 < r <= 0.50 and not math.isinf(r) and not math.isnan(r)]
        
        if valid_distances:
            if not self.safety_latch_engaged:
                closest_obstacle = min(valid_distances)
                self.get_logger().warn(f"🚨 TARGET DETECTED AT {closest_obstacle:.2f}m! SAFETY LATCH ENGAGED! 🚨")
                self.safety_latch_engaged = True
        else:
            if self.safety_latch_engaged:
                self.get_logger().info("✅ PATH CLEAR! Safety latch released.")
                self.safety_latch_engaged = False

def background_robot_executor():
    global global_robot_command, thread_active
    
    if not rclpy.ok():
        rclpy.init()
        
    robot = Robot() 
    node = LidarObserver(robot) 
    
    executor = rclpy.executors.SingleThreadedExecutor()
    executor.add_node(node)
    executor.add_node(robot)
    
    last_command = None
    last_command_time = time.time()
    
    print("🤖 Network Command Listener & LiDAR Arc Guard Fully Active.")

    try:
        while rclpy.ok() and thread_active:
            executor.spin_once(timeout_sec=0.01)

            current_cmd = global_robot_command
            current_time = time.time()

            if node.safety_latch_engaged and current_cmd == "GO":
                current_cmd = "STOP" 

            if current_cmd != last_command or (current_time - last_command_time) > 0.4:
                if current_cmd != last_command:
                    print(f"Server Executing Motion Matrix -> Command: {current_cmd}")
                
                if current_cmd in ["NO HAND DETECTED", "STOP", "EMERGENCY_STOP"]:
                    robot.set_cmd_vel(0.0, 0.0, 0.0)
                elif current_cmd == "GO":
                    robot.set_cmd_vel(0.2, 0.0, 0.5)
                elif current_cmd == "BACKWARD":
                    robot.set_cmd_vel(-0.2, 0.0, 0.5)
                elif current_cmd == "LEFT":
                    robot.set_cmd_vel(0.0, 0.2, 0.5)
                elif current_cmd == "RIGHT":
                    robot.set_cmd_vel(0.0, -0.2, 0.5)
                    
                last_command = current_cmd
                last_command_time = current_time 
                
            time.sleep(0.02) 

    except Exception as e:
        print(f"Thread encountered error: {e}")
    finally:
        executor.remove_node(node)
        executor.remove_node(robot)
        node.destroy_node()
        print("🛑 Robot background thread shut down cleanly.")

# ==========================================
# 4. GRADIO UI (Merged with Website Layout)
# ==========================================
custom_css = """
#Live-Feed {
    background-color: #D1BCE3;
}

.gradio-container {
    background-color: #D1BCE3;
}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# Hand Controlled Turtlebot Demo")
    gr.Markdown("### The Hand Signals That Control the Turtlebot Are Below:")
    gr.Markdown("The robot will stop moving when it does not detect any hand signals.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Gallery(
                value=[
                    "/Users/Dida/Desktop/Go.png",
                    "/Users/Dida/Desktop/Backward.png",
                ],
                label="Hand Signals (Forward / Backward)",
                columns=4,
                elem_id="hand-signals",
                height=220,
                show_label=False,
            )

            gr.Gallery(
                value=[
                    "/Users/Dida/Desktop/Left.png",
                    "/Users/Dida/Desktop/Right.png",
                ],
                label="Hand Signals (Left / Right)",
                columns=4,
                elem_id="hand-signals",
                height=220,
                show_label=False,
            )

        with gr.Column(scale=2):
            # We use gr.Image with streaming=True here so the frames pass into the Python backend
            live_feed_input = gr.Image(sources=["webcam"], streaming=True, visible=False)
            live_feed_output = gr.Image(label="Live Tracking Feed", elem_id="Live-Feed")
            
            # Pipe the live input frames through the OpenCV processing script
            live_feed_input.stream(fn=process_frame, inputs=live_feed_input, outputs=live_feed_output)

if __name__ == "__main__":
    # Start the ROS2 robot background thread
    worker_thread = threading.Thread(target=background_robot_executor)
    worker_thread.start()
    
    try:
        # Launch the web UI
        demo.launch(share=True)
    finally:
        # Gracefully shut down the robot thread when Gradio is closed via terminal (Ctrl+C)
        thread_active = False
        worker_thread.join()